# PSSP-ULTRA — Colab (stable recipe: early stopping + eps fix)

**Runtime → GPU.** The late-training divergence you hit was Adam's near-convergence blow-up; this version fixes it with a larger AdamW `eps`, tighter clip, and **early stopping** that ends training at the validation plateau (before the danger zone). The divergence guard remains as a last resort.

**Report one protocol.** Give each run its own `outdir` (the config uses a `RUN` tag) so runs never overwrite each other's `best.pt`, and report the stabilized run's best-val→CB513 number.

## 1. GPU

In [ ]:
!nvidia-smi -L
import torch; print('CUDA:', torch.cuda.is_available())

GPU 0: NVIDIA A100-SXM4-80GB (UUID: GPU-26657f31-62a0-4b6b-99b2-6a043908468a)
CUDA: True


## 2. Install

In [ ]:
!pip -q install pytorch-crf fair-esm

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 93.1/93.1 kB 10.8 MB/s eta 0:00:00


## 3. Mount Drive + locate data

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
DATA_DIR = '/content/drive/MyDrive/PSSP_ULTRA_DATASETS'
import os
TRAIN = os.path.join(DATA_DIR, 'cullpdb+profile_5926_updated.npy.gz')
TEST  = os.path.join(DATA_DIR, 'cb513+profile_split1_updated.npy.gz')
assert os.path.exists(TRAIN) and os.path.exists(TEST), 'check DATA_DIR / names'
print('found both files.')

Mounted at /content/drive
found both files.


## 4. Pipeline

In [ ]:
#!/usr/bin/env python
# -*- coding: utf-8 -*-
"""
PSSP-ULTRA : training pipeline for the canonical Princeton / Zhou-Troyanskaya files
===================================================================================
Inputs (your uploaded files):
    cullpdb_profile_5926_updated_npy.gz   -> (5926, 700, 57)   [train + internal val]
    cb513_profile_split1_updated_npy.gz   -> ( 514, 700, 57)   [held-out test]

Model: reconstruct sequence from one-hot -> ESM-2 per-residue embedding (cached)
       concat[ ESM-2 || profile || one-hot ] -> proj -> TCN -> BiLSTM
       -> Pre-LN Transformer -> CRF (decodes Q8; Q3 via standard 8->3 collapse).

CHECKPOINTING (new):
    * A full training checkpoint (model + optimiser + scheduler + RNG + best-so-far)
      is written to <outdir>/ckpt_last.pt every `save_every` epochs (default 5),
      atomically (temp file + os.replace) so a mid-write disconnect cannot corrupt it.
    * On start, if <outdir>/ckpt_last.pt exists, training RESUMES from the saved epoch.
      Put outdir on Google Drive so it survives a Colab disconnect: just re-run and it
      continues where it left off.
    * torch.load uses weights_only=False (the file is your own, trusted checkpoint;
      PyTorch >=2.6 defaults weights_only=True and would otherwise refuse to load it).

RUN:
    pip install torch numpy scikit-learn pytorch-crf fair-esm tqdm
    python train_pssp_ultra.py --train cullpdb_profile_5926_updated_npy.gz \
        --test cb513_profile_split1_updated_npy.gz \
        --outdir /content/drive/MyDrive/PSSP/runs/ultra_esm650 --save_every 5

Numbers are produced by YOUR run. Nothing here hard-codes an accuracy.
"""

import os, gzip, json, time, hashlib, argparse, random
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

# ---------------------------------------------------------------------------
# 0. Repro + config + constants
# ---------------------------------------------------------------------------
def set_seed(s=42):
    random.seed(s); np.random.seed(s)
    torch.manual_seed(s); torch.cuda.manual_seed_all(s)

AA_ORDER = list("ACEDGFIHKMLNPQRSTWVY") + ["X", "NoSeq"]        # cols [0:22)
SS8_ORDER = ["L", "B", "E", "G", "I", "H", "S", "T", "NoSeq"]   # cols [22:31)
Q8_TO_Q3 = {"H": "H", "G": "H", "I": "H", "E": "E", "B": "E",
            "L": "C", "S": "C", "T": "C"}
SS3_ORDER = ["H", "E", "C"]
SS3_IDX = {s: i for i, s in enumerate(SS3_ORDER)}
Q8IDX_TO_Q3IDX = np.array([SS3_IDX[Q8_TO_Q3[SS8_ORDER[i]]] for i in range(8)], dtype=np.int64)

MAXLEN = 700
ONEHOT = slice(0, 22); SSLAB = slice(22, 31); PROFILE = slice(35, 57)


# ---------------------------------------------------------------------------
# 1. Data loading
# ---------------------------------------------------------------------------
def load_npy_gz(path):
    with gzip.open(path, "rb") as fh:
        arr = np.load(fh, allow_pickle=True)
    return arr.reshape(-1, MAXLEN, 57).astype(np.float32)

def decode_sequences(data):
    oh = data[:, :, ONEHOT]; idx = oh.argmax(-1)
    noseq = AA_ORDER.index("NoSeq")
    valid = (idx != noseq) & (oh.sum(-1) > 0.5)
    seqs = []
    for b in range(data.shape[0]):
        L = int(valid[b].sum())
        seqs.append("".join(AA_ORDER[idx[b, t]] for t in range(L)))
    return seqs, valid

def q8_q3_labels(data, valid):
    ss = data[:, :, SSLAB]
    y8 = ss[:, :, :8].argmax(-1).astype(np.int64); y8[~valid] = -100
    y3 = np.where(valid, Q8IDX_TO_Q3IDX[np.clip(y8, 0, 7)], -100).astype(np.int64)
    return y8, y3


# ---------------------------------------------------------------------------
# 2. ESM-2 embeddings, cached to a versioned memmap
# ---------------------------------------------------------------------------
def esm_cache(seqs, split_name, model_name, cache_dir, device, batch=8):
    """Precompute & cache ESM-2 last-layer embeddings -> float16 memmap (N,700,D)."""
    import esm
    os.makedirs(cache_dir, exist_ok=True)
    tag = hashlib.md5((model_name + split_name + str(len(seqs))).encode()).hexdigest()[:8]
    cache = os.path.join(cache_dir, f"esm_{split_name}_{tag}.npy"); meta = cache + ".json"
    model, alphabet = esm.pretrained.load_model_and_alphabet(model_name)
    D = model.embed_dim
    if os.path.exists(cache) and os.path.exists(meta):
        m = json.load(open(meta))
        if m["n"] == len(seqs) and m["D"] == D and m["model"] == model_name:
            print(f"[esm] using cache {cache}  (D={D})")
            return np.load(cache, mmap_mode="r"), D
    print(f"[esm] computing {split_name} embeddings with {model_name} (D={D}) ...")
    model = model.to(device).eval()
    for p in model.parameters(): p.requires_grad_(False)
    bc = alphabet.get_batch_converter(); last = model.num_layers
    mm = np.lib.format.open_memmap(cache, mode="w+", dtype=np.float16, shape=(len(seqs), MAXLEN, D))
    with torch.no_grad():
        for i in range(0, len(seqs), batch):
            chunk = [(str(j), seqs[i + j]) for j in range(min(batch, len(seqs) - i))]
            _, _, toks = bc(chunk); toks = toks.to(device)
            out = model(toks, repr_layers=[last])["representations"][last]
            for j, (_, s) in enumerate(chunk):
                L = len(s)
                mm[i + j, :L] = out[j, 1:L + 1, :].float().cpu().numpy().astype(np.float16)
    mm.flush(); json.dump({"n": len(seqs), "D": D, "model": model_name}, open(meta, "w"))
    del model; torch.cuda.empty_cache()
    return np.load(cache, mmap_mode="r"), D


# ---------------------------------------------------------------------------
# 3. Dataset
# ---------------------------------------------------------------------------
class PSSPSet(Dataset):
    def __init__(self, data, valid, y8, y3, esm_mm, use_esm):
        self.oh = data[:, :, ONEHOT]; self.prof = data[:, :, PROFILE]
        self.valid, self.y8, self.y3 = valid, y8, y3
        self.esm, self.use_esm = esm_mm, use_esm
    def __len__(self): return self.oh.shape[0]
    def __getitem__(self, i):
        feats = [self.oh[i], self.prof[i]]
        if self.use_esm: feats.append(np.asarray(self.esm[i], dtype=np.float32))
        x = np.concatenate(feats, axis=-1)
        return (torch.from_numpy(x).float(),
                torch.from_numpy(self.valid[i].astype(np.bool_)),
                torch.from_numpy(self.y8[i]).long(),
                torch.from_numpy(self.y3[i]).long())


# ---------------------------------------------------------------------------
# 4. Model
# ---------------------------------------------------------------------------
class TCNBlock(nn.Module):
    def __init__(self, c, k=3, d=1, p=0.1):
        super().__init__(); pad = (k - 1) * d // 2
        self.net = nn.Sequential(
            nn.Conv1d(c, c, k, padding=pad, dilation=d), nn.GELU(), nn.Dropout(p),
            nn.Conv1d(c, c, k, padding=pad, dilation=d), nn.GELU(), nn.Dropout(p))
        self.norm = nn.LayerNorm(c)
    def forward(self, x):
        h = self.net(x.transpose(1, 2)).transpose(1, 2)
        return self.norm(x + h)

class PSSPUltra(nn.Module):
    def __init__(self, in_dim, d_model=256, n_tcn=3, n_tr=2, heads=8, n_cls=8, p=0.2,
                 use_tcn=True, use_transformer=True, use_crf=True):
        super().__init__()
        self.use_tcn, self.use_transformer, self.use_crf = use_tcn, use_transformer, use_crf
        self.proj = nn.Sequential(nn.Linear(in_dim, d_model), nn.GELU(), nn.Dropout(p))
        if use_tcn:
            self.tcn = nn.ModuleList([TCNBlock(d_model, d=2 ** i, p=p) for i in range(n_tcn)])
        self.lstm = nn.LSTM(d_model, d_model // 2, num_layers=2, batch_first=True,
                            bidirectional=True, dropout=p)
        if use_transformer:
            enc = nn.TransformerEncoderLayer(d_model, heads, d_model * 4, p, activation="gelu",
                                             batch_first=True, norm_first=True)
            self.tr = nn.TransformerEncoder(enc, n_tr)
        self.out = nn.Linear(d_model, n_cls)
        if use_crf:
            from torchcrf import CRF
            self.crf = CRF(n_cls, batch_first=True)
    def emissions(self, x, valid):
        h = self.proj(x)
        if self.use_tcn:
            for blk in self.tcn: h = blk(h)
        h, _ = self.lstm(h)
        if self.use_transformer:
            h = self.tr(h, src_key_padding_mask=~valid)
        return self.out(h)
    def loss(self, x, valid, y8):
        em = self.emissions(x, valid)
        if self.use_crf:
            y = y8.clone(); y[~valid] = 0
            return -self.crf(em, y, mask=valid, reduction="mean")
        # ablation: plain masked cross-entropy (padded labels are -100)
        return F.cross_entropy(em.reshape(-1, em.size(-1)), y8.reshape(-1), ignore_index=-100)
    @torch.no_grad()
    def decode(self, x, valid):
        em = self.emissions(x, valid)
        if self.use_crf:
            return self.crf.decode(em, mask=valid)
        pred = em.argmax(-1)                                     # ablation: greedy argmax
        return [pred[i, :int(valid[i].sum())].tolist() for i in range(pred.size(0))]


# ---------------------------------------------------------------------------
# 5. Metrics
# ---------------------------------------------------------------------------
def q_accuracy(pred8, y8, valid):
    correct = tot = correct3 = 0
    for p, yt, m in zip(pred8, y8, valid):
        L = int(m.sum()); p = np.asarray(p[:L]); g = yt[:L].numpy()
        correct += (p == g).sum(); tot += L
        correct3 += (Q8IDX_TO_Q3IDX[p] == Q8IDX_TO_Q3IDX[g]).sum()
    return correct3 / tot, correct / tot

def sov(pred3, gt3):
    def segs(a):
        out, s = [], 0
        for i in range(1, len(a) + 1):
            if i == len(a) or a[i] != a[s]: out.append((a[s], s, i - 1)); s = i
        return out
    sp, sg = segs(pred3), segs(gt3); num = den = 0.0
    for cls in range(3):
        gs = [x for x in sg if x[0] == cls]; ps = [x for x in sp if x[0] == cls]
        for (_, gb, ge) in gs:
            overlap = [(pb, pe) for (_, pb, pe) in ps if pe >= gb and pb <= ge]
            lg = ge - gb + 1
            if not overlap: den += lg; continue
            for (pb, pe) in overlap:
                minov = min(ge, pe) - max(gb, pb) + 1
                maxov = max(ge, pe) - min(gb, pb) + 1
                lp = pe - pb + 1
                delta = min(maxov - minov, minov, lg // 2 if lg // 2 else 1,
                            lp // 2 if lp // 2 else 1)
                num += (minov + delta) / maxov * lg; den += lg
    return 100.0 * num / den if den else 0.0

def sov_dataset(pred8, y8, valid):
    tot, n = 0.0, 0
    for p, yt, m in zip(pred8, y8, valid):
        L = int(m.sum())
        tot += sov(Q8IDX_TO_Q3IDX[np.asarray(p[:L])], Q8IDX_TO_Q3IDX[yt[:L].numpy()]); n += 1
    return tot / max(n, 1)

def evaluate(model, loader, device):
    model.eval(); P, Y, M = [], [], []
    with torch.no_grad():
        for x, valid, y8, _ in loader:
            P += model.decode(x.to(device), valid.to(device)); Y += list(y8); M += list(valid)
    q3, q8 = q_accuracy(P, Y, M)
    return q3, q8, sov_dataset(P, Y, M)


# ---------------------------------------------------------------------------
# 6. Checkpoint helpers (atomic save + RNG capture)
# ---------------------------------------------------------------------------
def _atomic_save(obj, path):
    """Write to a temp file then rename, so a disconnect mid-write can't corrupt path."""
    tmp = path + ".tmp"
    torch.save(obj, tmp)
    try:
        os.replace(tmp, path)                 # atomic on the same filesystem
    except OSError:                           # some FUSE mounts dislike replacing an open target
        if os.path.exists(path): os.remove(path)
        os.replace(tmp, path)

def _rng_state():
    st = {"torch": torch.get_rng_state(), "np": np.random.get_state(), "py": random.getstate()}
    if torch.cuda.is_available(): st["cuda"] = torch.cuda.get_rng_state_all()
    return st

def _set_rng_state(st):
    try:
        if "torch" in st: torch.set_rng_state(st["torch"].cpu().to(torch.uint8))
        if "np" in st: np.random.set_state(st["np"])
        if "py" in st: random.setstate(st["py"])
        if "cuda" in st and torch.cuda.is_available():
            torch.cuda.set_rng_state_all([t.cpu().to(torch.uint8) for t in st["cuda"]])
    except Exception as e:
        print("[resume] RNG restore skipped:", e)


# ---------------------------------------------------------------------------
# 6b. Optimizer with param groups (stability)
#     - no weight decay on biases / LayerNorm / CRF transitions
#     - CRF gets a reduced learning rate (it is the layer that blows up first)
# ---------------------------------------------------------------------------
def build_optimizer(model, lr, wd, crf_scale, eps):
    decay, no_decay, crf = [], [], []
    for n, p in model.named_parameters():
        if not p.requires_grad: continue
        if n.startswith("crf."):                              crf.append(p)
        elif p.ndim == 1 or n.endswith(".bias") or "norm" in n.lower(): no_decay.append(p)
        else:                                                 decay.append(p)
    groups, max_lrs = [], []
    for params, wdecay, l in [(decay, wd, lr), (no_decay, 0.0, lr), (crf, 0.0, lr * crf_scale)]:
        if params:                                            # skip empty groups (e.g. no CRF)
            groups.append({"params": params, "weight_decay": wdecay, "lr": l}); max_lrs.append(l)
    opt = torch.optim.AdamW(groups, lr=lr, eps=eps)          # larger eps stabilises the denominator
    return opt, max_lrs


# ---------------------------------------------------------------------------
# 7. Train / evaluate with checkpoint + resume
# ---------------------------------------------------------------------------
def main(a):
    set_seed(a.seed)
    dev = "cuda" if torch.cuda.is_available() else "cpu"
    os.makedirs(a.outdir, exist_ok=True)
    cache_dir = a.cache_dir or a.outdir          # embeddings persist here too (see note in run cell)

    tr = load_npy_gz(a.train); te = load_npy_gz(a.test)
    tr_seq, tr_valid = decode_sequences(tr); te_seq, te_valid = decode_sequences(te)
    tr_y8, tr_y3 = q8_q3_labels(tr, tr_valid); te_y8, te_y3 = q8_q3_labels(te, te_valid)

    tr_esm = te_esm = None; edim = 0
    if a.use_esm:
        tr_esm, edim = esm_cache(tr_seq, "train", a.esm_model, cache_dir, dev, a.esm_batch)
        te_esm, _    = esm_cache(te_seq, "cb513", a.esm_model, cache_dir, dev, a.esm_batch)

    n = tr.shape[0]; perm = np.random.RandomState(a.seed).permutation(n)
    vi, ti = perm[:a.val_size], perm[a.val_size:]
    sub = lambda arr, idx: None if arr is None else arr[idx]
    trainset = PSSPSet(tr[ti], tr_valid[ti], tr_y8[ti], tr_y3[ti], sub(tr_esm, ti), a.use_esm)
    valset   = PSSPSet(tr[vi], tr_valid[vi], tr_y8[vi], tr_y3[vi], sub(tr_esm, vi), a.use_esm)
    testset  = PSSPSet(te, te_valid, te_y8, te_y3, te_esm, a.use_esm)
    dl = lambda ds, sh: DataLoader(ds, batch_size=a.batch, shuffle=sh, num_workers=2)
    tl, vl, xl = dl(trainset, True), dl(valset, False), dl(testset, False)

    in_dim = 22 + 22 + (edim if a.use_esm else 0)
    model = PSSPUltra(in_dim, a.d_model, a.n_tcn, a.n_tr, a.heads, 8, a.dropout,
                      use_tcn=getattr(a, "use_tcn", True),
                      use_transformer=getattr(a, "use_transformer", True),
                      use_crf=getattr(a, "use_crf", True)).to(dev)
    opt, max_lrs = build_optimizer(model, a.lr, a.wd, a.crf_lr_scale, a.eps)
    steps = a.epochs * len(tl)
    sched = torch.optim.lr_scheduler.OneCycleLR(opt, max_lr=max_lrs, total_steps=steps, pct_start=0.1)
    print(f"[model] in_dim={in_dim}  params={sum(p.numel() for p in model.parameters())/1e6:.2f}M")

    # ----- resume from Drive checkpoint if present -----
    ckpt_path = os.path.join(a.outdir, "ckpt_last.pt")
    best_path = os.path.join(a.outdir, "best.pt")
    prog_path = os.path.join(a.outdir, "progress.json")
    start_epoch, best_val, best, history = 1, -1.0, None, []
    since_improve = 0                          # epochs since last validation improvement

    if os.path.exists(ckpt_path):
        print(f"[resume] found {ckpt_path} -> loading")
        ck = torch.load(ckpt_path, map_location=dev, weights_only=False)   # trusted, our own file
        if ck.get("total_steps") != steps:
            print(f"[resume] WARNING: schedule length changed ({ck.get('total_steps')} -> {steps}). "
                  f"For a clean resume keep epochs/batch/val_size/use_esm unchanged.")
        model.load_state_dict(ck["model"])
        try: opt.load_state_dict(ck["opt"])
        except Exception as e: print("[resume] optimizer restore skipped (fresh optimizer):", e)
        try: sched.load_state_dict(ck["sched"])
        except Exception as e: print("[resume] sched restore skipped:", e)
        start_epoch = ck["epoch"] + 1; best_val = ck["best_val"]
        best = ck["best_state"]; history = ck.get("history", [])
        _set_rng_state(ck.get("rng", {}))
        print(f"[resume] continuing from epoch {start_epoch}/{a.epochs} "
              f"(best val Q3 so far {best_val*100:.2f})")
    else:
        print("[resume] no checkpoint -> training from scratch")

    # ----- training loop -----
    for ep in range(start_epoch, a.epochs + 1):
        model.train(); t0 = time.time(); tot = 0.0; nb = 0; skipped = 0
        for x, valid, y8, _ in tl:
            x, valid, y8 = x.to(dev), valid.to(dev), y8.to(dev)
            opt.zero_grad(set_to_none=True)
            loss = model.loss(x, valid, y8)
            if not torch.isfinite(loss):                 # skip a bad batch instead of poisoning weights
                skipped += 1; continue
            loss.backward()
            gnorm = nn.utils.clip_grad_norm_(model.parameters(), a.clip)
            if not torch.isfinite(gnorm):                # non-finite gradient -> skip the step
                skipped += 1; continue
            opt.step(); sched.step(); tot += loss.item(); nb += 1
        avg = tot / max(nb, 1)
        vq3, vq8, vsov = evaluate(model, vl, dev)
        history.append({"epoch": ep, "val_Q3": round(vq3 * 100, 3),
                        "val_Q8": round(vq8 * 100, 3), "val_SOV": round(vsov, 3)})
        print(f"ep{ep:03d} loss{avg:8.3f}  val Q3 {vq3*100:5.2f}  Q8 {vq8*100:5.2f}  "
              f"SOV {vsov:5.2f}  ({time.time()-t0:.0f}s)" + (f"  [skipped {skipped}]" if skipped else ""))

        if vq3 > best_val:
            best_val = vq3
            best = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
            _atomic_save(best, best_path)
            since_improve = 0
        else:
            since_improve += 1

        # divergence guard: loss non-finite, or validation collapsed far below best
        diverged = (not np.isfinite(avg)) or (best_val > 0 and vq3 < best_val - 0.20)
        if diverged:
            print(f"   [guard] divergence detected (avg loss {avg:.1f}, val Q3 {vq3*100:.2f} "
                  f"vs best {best_val*100:.2f}). Stopping; best.pt is preserved. "
                  f"Lower --lr (try 3e-4) and rerun.")
            break

        # periodic + final checkpoint to Drive (only while healthy)
        if ep % a.save_every == 0 or ep == a.epochs:
            _atomic_save({"epoch": ep, "model": model.state_dict(), "opt": opt.state_dict(),
                          "sched": sched.state_dict(), "best_val": best_val, "best_state": best,
                          "history": history, "rng": _rng_state(), "total_steps": steps,
                          "cfg": {"epochs": a.epochs, "batch": a.batch,
                                  "val_size": a.val_size, "use_esm": a.use_esm}}, ckpt_path)
            json.dump(history, open(prog_path, "w"), indent=2)
            print(f"   [ckpt] saved through epoch {ep} -> {ckpt_path}")

        # early stopping: end cleanly at the plateau, before the late-training divergence zone
        if a.patience > 0 and since_improve >= a.patience:
            print(f"   [early-stop] no val improvement for {a.patience} epochs "
                  f"(best Q3 {best_val*100:.2f} at epoch {ep - since_improve}). Stopping cleanly.")
            break

    # ----- final held-out evaluation with best checkpoint -----
    if best is None:
        best = (torch.load(best_path, map_location=dev, weights_only=False)
                if os.path.exists(best_path) else model.state_dict())
    model.load_state_dict(best)
    q3, q8, sv = evaluate(model, xl, dev)
    res = {"CB513_Q3": round(q3 * 100, 2), "CB513_Q8": round(q8 * 100, 2),
           "CB513_SOV": round(sv, 2), "val_Q3": round(best_val * 100, 2),
           "use_esm": a.use_esm, "esm_model": a.esm_model if a.use_esm else None}
    json.dump(res, open(os.path.join(a.outdir, "cb513_results.json"), "w"), indent=2)
    print("\n=== CB513 (held-out) ===\n" + json.dumps(res, indent=2))
    print("\nThese are the numbers to place in the manuscript results table.")
    return res


## 5. Configuration
`RUN` names the output folder — bump it for a new configuration so nothing is overwritten. Stable defaults: `lr=5e-4`, `eps=1e-6`, `crf_lr_scale=0.1`, `clip=0.5`, `patience=10`.

In [ ]:
from types import SimpleNamespace
RUN = 'stable_v1'                     # <-- change for each new configuration/seed
CONFIG = dict(
    train      = TRAIN,
    test       = TEST,
    outdir     = f'/content/drive/MyDrive/PSSP_ULTRA_DATASETS/runs/{RUN}',
    cache_dir  = '/content/drive/MyDrive/PSSP_ULTRA_DATASETS/esm_cache',
    save_every = 5,
    use_esm    = True,
    esm_model  = 'esm2_t33_650M_UR50D',
    esm_batch  = 4,
    epochs     = 60,
    batch      = 16,
    val_size   = 300,
    d_model = 256, n_tcn = 3, n_tr = 2, heads = 8, dropout = 0.2,
    lr = 5e-4, crf_lr_scale = 0.1, clip = 0.5, eps = 1e-6,
    patience = 10, wd = 1e-2, seed = 42,
)
os.makedirs(CONFIG['outdir'], exist_ok=True)
CONFIG

{'train': '/content/drive/MyDrive/PSSP_ULTRA_DATASETS/cullpdb+profile_5926_updated.npy.gz',
 'test': '/content/drive/MyDrive/PSSP_ULTRA_DATASETS/cb513+profile_split1_updated.npy.gz',
 'outdir': '/content/drive/MyDrive/PSSP_ULTRA_DATASETS/runs/stable_v1',
 'cache_dir': '/content/drive/MyDrive/PSSP_ULTRA_DATASETS/esm_cache',
 'save_every': 5,
 'use_esm': True,
 'esm_model': 'esm2_t33_650M_UR50D',
 'esm_batch': 4,
 'epochs': 60,
 'batch': 16,
 'val_size': 300,
 'd_model': 256,
 'n_tcn': 3,
 'n_tr': 2,
 'heads': 8,
 'dropout': 0.2,
 'lr': 0.0005,
 'crf_lr_scale': 0.1,
 'clip': 0.5,
 'eps': 1e-06,
 'patience': 10,
 'wd': 0.01,
 'seed': 42}

## 6. Train (stable, early-stopped)
Ends cleanly at the plateau. With a fresh `RUN` folder there is no old checkpoint to resume, so this is a clean single-protocol run — the number to report.

In [ ]:
main(SimpleNamespace(**CONFIG))

[esm] using cache /content/drive/MyDrive/PSSP_ULTRA_DATASETS/esm_cache/esm_train_99ebc664.npy  (D=1280)
[esm] using cache /content/drive/MyDrive/PSSP_ULTRA_DATASETS/esm_cache/esm_cb513_4f9e0b20.npy  (D=1280)
[model] in_dim=1324  params=3.89M
[resume] found /content/drive/MyDrive/PSSP_ULTRA_DATASETS/runs/stable_v1/ckpt_last.pt -> loading
[resume] continuing from epoch 6/60 (best val Q3 so far 86.78)


/tmp/ipykernel_18085/3167966161.py:159: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.norm_first was True
  self.tr = nn.TransformerEncoder(enc, n_tr)


ep006 loss 125.599  val Q3 87.10  Q8 77.30  SOV 85.86  (290s)
ep007 loss 120.037  val Q3 87.76  Q8 77.96  SOV 86.36  (290s)
ep008 loss 114.617  val Q3 87.48  Q8 77.66  SOV 85.47  (291s)
ep009 loss 109.770  val Q3 87.09  Q8 77.87  SOV 85.63  (291s)
ep010 loss 105.164  val Q3 87.75  Q8 78.16  SOV 86.22  (291s)
   [ckpt] saved through epoch 10 -> /content/drive/MyDrive/PSSP_ULTRA_DATASETS/runs/stable_v1/ckpt_last.pt
ep011 loss 100.915  val Q3 87.45  Q8 78.02  SOV 85.99  (290s)
ep012 loss  97.135  val Q3 87.59  Q8 77.90  SOV 85.82  (290s)
ep013 loss  93.690  val Q3 87.03  Q8 77.81  SOV 85.52  (291s)
ep014 loss  90.589  val Q3 87.41  Q8 77.83  SOV 85.97  (290s)
ep015 loss  87.834  val Q3 87.61  Q8 77.80  SOV 86.06  (290s)
   [ckpt] saved through epoch 15 -> /content/drive/MyDrive/PSSP_ULTRA_DATASETS/runs/stable_v1/ckpt_last.pt
ep016 loss  85.221  val Q3 87.18  Q8 77.90  SOV 85.85  (291s)
ep017 loss  82.804  val Q3 87.50  Q8 78.00  SOV 86.28  (290s)
   [early-stop] no val improvement for 10 

{'CB513_Q3': np.float64(86.54),
 'CB513_Q8': np.float64(75.71),
 'CB513_SOV': 83.83,
 'val_Q3': np.float64(87.76),
 'use_esm': True,
 'esm_model': 'esm2_t33_650M_UR50D'}

## 6b. Evaluate `best.pt` on CB513 (any time, no training)

In [ ]:
import torch, os
from torch.utils.data import DataLoader
dev = 'cuda' if torch.cuda.is_available() else 'cpu'
od  = CONFIG['outdir']
te  = load_npy_gz(CONFIG['test']); te_seq, te_valid = decode_sequences(te)
te_y8, te_y3 = q8_q3_labels(te, te_valid)
cache_dir = CONFIG['cache_dir'] or od
te_esm, edim = (esm_cache(te_seq,'cb513',CONFIG['esm_model'],cache_dir,dev,CONFIG['esm_batch'])
                if CONFIG['use_esm'] else (None,0))
xl = DataLoader(PSSPSet(te, te_valid, te_y8, te_y3, te_esm, CONFIG['use_esm']), batch_size=CONFIG['batch'])
in_dim = 22 + 22 + (edim if CONFIG['use_esm'] else 0)
model = PSSPUltra(in_dim, CONFIG['d_model'], CONFIG['n_tcn'], CONFIG['n_tr'],
                  CONFIG['heads'], 8, CONFIG['dropout']).to(dev)
model.load_state_dict(torch.load(os.path.join(od,'best.pt'), map_location=dev, weights_only=False))
q3,q8,sv = evaluate(model, xl, dev)
print(f'CB513 (best.pt):  Q3 {q3*100:.2f}   Q8 {q8*100:.2f}   SOV {sv:.2f}')

Downloading: "https://dl.fbaipublicfiles.com/fair-esm/models/esm2_t33_650M_UR50D.pt" to /root/.cache/torch/hub/checkpoints/esm2_t33_650M_UR50D.pt
Downloading: "https://dl.fbaipublicfiles.com/fair-esm/regression/esm2_t33_650M_UR50D-contact-regression.pt" to /root/.cache/torch/hub/checkpoints/esm2_t33_650M_UR50D-contact-regression.pt
[esm] using cache /content/drive/MyDrive/PSSP_ULTRA_DATASETS/esm_cache/esm_cb513_4f9e0b20.npy  (D=1280)


/tmp/ipykernel_9437/3167966161.py:159: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.norm_first was True
  self.tr = nn.TransformerEncoder(enc, n_tr)


CB513 (best.pt):  Q3 86.54   Q8 75.71   SOV 83.83


## 7. Profile-only ablation (REQUIRED for the paper)
Same architecture, `use_esm=False`. This is what proves the ESM-2 embedding — not the encoder — drives the gain. Its own folder, so it won't touch the full-model checkpoints.

In [ ]:
import copy
c = copy.deepcopy(CONFIG); c['use_esm'] = False
c['outdir'] = f"/content/drive/MyDrive/PSSP_ULTRA_DATASETS/runs/{RUN}_noesm"
os.makedirs(c['outdir'], exist_ok=True)
print('profile-only:', main(SimpleNamespace(**c)))

/tmp/ipykernel_653/3167966161.py:159: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.norm_first was True
  self.tr = nn.TransformerEncoder(enc, n_tr)


[model] in_dim=44  params=3.57M
[resume] found /content/drive/MyDrive/PSSP_ULTRA_DATASETS/runs/stable_v1_noesm/ckpt_last.pt -> loading
[resume] continuing from epoch 26/60 (best val Q3 so far 83.84)
ep026 loss 103.136  val Q3 83.52  Q8 72.23  SOV 82.31  (279s)
ep027 loss 101.462  val Q3 83.35  Q8 72.20  SOV 81.79  (280s)
ep028 loss  99.705  val Q3 83.76  Q8 72.15  SOV 82.43  (280s)
ep029 loss  98.165  val Q3 83.28  Q8 71.98  SOV 81.64  (280s)
ep030 loss  96.689  val Q3 83.53  Q8 72.25  SOV 82.26  (281s)
   [ckpt] saved through epoch 30 -> /content/drive/MyDrive/PSSP_ULTRA_DATASETS/runs/stable_v1_noesm/ckpt_last.pt
ep031 loss  95.199  val Q3 83.77  Q8 72.20  SOV 82.33  (280s)
ep032 loss  93.689  val Q3 83.88  Q8 72.34  SOV 82.49  (279s)
ep033 loss  92.179  val Q3 83.76  Q8 72.20  SOV 81.83  (280s)
ep034 loss  90.851  val Q3 83.33  Q8 71.99  SOV 82.14  (278s)
ep035 loss  89.622  val Q3 83.70  Q8 72.44  SOV 82.02  (279s)
   [ckpt] saved through epoch 35 -> /content/drive/MyDrive/PSSP_ULTR

## 8. Multi-seed (5 seeds): full vs profile-only + paired significance
Runs **both** conditions across 5 seeds. Completed seeds load instantly from their saved `cb513_results.json`; only new seeds train (~40 min each). Prints mean ± SD (sample, n−1) and a **paired t-test** of the ESM-2 gain per metric — the extra seeds push Q3/Q8 under 0.05.

In [ ]:
import copy, os, json, numpy as np
from types import SimpleNamespace

SEEDS = [42, 1, 7, 2, 3]              # first three done; 2 and 3 are new
base  = CONFIG['outdir']

def run_or_load(use_esm, seed):
    od = f'{base}_seed{seed}' if use_esm else f'{base}_noesm_seed{seed}'
    rp = os.path.join(od, 'cb513_results.json'); tag = 'full   ' if use_esm else 'profile'
    if os.path.exists(rp):
        print(f'  [load] {tag} seed{seed}'); return json.load(open(rp))
    print(f'  [run ] {tag} seed{seed}')
    c = copy.deepcopy(CONFIG); c['use_esm']=use_esm; c['seed']=seed; c['outdir']=od
    os.makedirs(od, exist_ok=True); return main(SimpleNamespace(**c))

mets=['CB513_Q3','CB513_Q8','CB513_SOV']; full={m:[] for m in mets}; prof={m:[] for m in mets}
for s in SEEDS:
    rf=run_or_load(True,s); rp=run_or_load(False,s)
    for m in mets: full[m].append(rf[m]); prof[m].append(rp[m])

try:
    from scipy import stats; have=True
except Exception:
    have=False; print('scipy missing -> p skipped (!pip install scipy)')
lab={'CB513_Q3':'Q3','CB513_Q8':'Q8','CB513_SOV':'SOV'}
print(f'\nn = {len(SEEDS)} seeds {SEEDS}')
print(f"{'metric':7}{'full (ESM-2)':>16}{'profile-only':>16}{'d(ESM)':>9}{'paired p':>11}{'dz':>7}")
summary={}
for m in mets:
    a=np.array(full[m],float); b=np.array(prof[m],float); d=a-b; sd=lambda v:v.std(ddof=1)
    p=float(stats.ttest_rel(a,b)[1]) if have else float('nan')
    dz=d.mean()/sd(d) if sd(d)>0 else float('nan')
    summary[m]=dict(full_mean=a.mean(),full_sd=sd(a),prof_mean=b.mean(),prof_sd=sd(b),delta=d.mean(),p=p,dz=dz)
    print(f"{lab[m]:7}{a.mean():8.2f} ±{sd(a):4.2f}{b.mean():9.2f} ±{sd(b):4.2f}{d.mean():+8.2f}{p:11.4f}{dz:7.2f}")
print('\nper-seed Q3 full:', [round(x,2) for x in full['CB513_Q3']])
print('per-seed Q3 prof:', [round(x,2) for x in prof['CB513_Q3']])
json.dump({'seeds':SEEDS,'full':full,'profile':prof,'summary':summary}, open(os.path.join(base,'multiseed_summary.json'),'w'), indent=2)
print('\nsaved', os.path.join(base,'multiseed_summary.json'), '- send me this to fill the paper.')

  [load] full    seed42
  [load] profile seed42
  [load] full    seed1
  [load] profile seed1
  [load] full    seed7
  [load] profile seed7
  [load] full    seed2
  [load] profile seed2
  [load] full    seed3
  [run ] profile seed3


/tmp/ipykernel_843/2477223729.py:164: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.norm_first was True
  self.tr = nn.TransformerEncoder(enc, n_tr)


[model] in_dim=44  params=3.57M
[resume] found /content/drive/MyDrive/PSSP_ULTRA_DATASETS/runs/stable_v1_noesm_seed3/ckpt_last.pt -> loading
[resume] continuing from epoch 11/60 (best val Q3 so far 84.06)
ep011 loss 141.804  val Q3 83.26  Q8 72.25  SOV 82.42  (264s)
ep012 loss 138.331  val Q3 83.34  Q8 72.15  SOV 82.55  (264s)
ep013 loss 134.840  val Q3 83.79  Q8 72.32  SOV 82.40  (264s)
ep014 loss 131.819  val Q3 83.58  Q8 72.65  SOV 82.50  (263s)
ep015 loss 128.554  val Q3 83.95  Q8 72.80  SOV 83.05  (263s)
   [ckpt] saved through epoch 15 -> /content/drive/MyDrive/PSSP_ULTRA_DATASETS/runs/stable_v1_noesm_seed3/ckpt_last.pt
ep016 loss 125.834  val Q3 84.13  Q8 72.82  SOV 83.45  (263s)
ep017 loss 123.135  val Q3 83.87  Q8 72.54  SOV 82.90  (265s)
ep018 loss 120.765  val Q3 83.72  Q8 72.39  SOV 83.17  (264s)
ep019 loss 118.146  val Q3 83.59  Q8 72.91  SOV 82.16  (265s)
ep020 loss 115.970  val Q3 84.16  Q8 72.97  SOV 83.13  (263s)
   [ckpt] saved through epoch 20 -> /content/drive/MyDri

## 9. Architecture ablations (− CRF, − Transformer, − TCN)
Single-seed (default CONFIG, ESM-2 on); each removes one component, own folder. Also loads the seed-matched Full and − ESM-2 rows so the whole ablation table prints together. ~40 min each.

In [ ]:
import copy, os, json
from types import SimpleNamespace
ablate=[('noCRF',dict(use_crf=False)),('noTransformer',dict(use_transformer=False)),('noTCN',dict(use_tcn=False))]
res={}
for tag,over in ablate:
    c=copy.deepcopy(CONFIG); c.update(over); c['outdir']=CONFIG['outdir']+'_'+tag
    os.makedirs(c['outdir'],exist_ok=True); print('\n=== ablation:',tag,'==='); res[tag]=main(SimpleNamespace(**c))
def load(od):
    p=os.path.join(od,'cb513_results.json'); return json.load(open(p)) if os.path.exists(p) else None
full_ref=load(CONFIG['outdir']); noesm=load(CONFIG['outdir']+'_noesm')
print('\n=========  Ablation table (CB513, single seed)  =========')
print(f"{'variant':22}{'Q3':>8}{'Q8':>8}{'SOV':>8}")
def row(n,r):
    print(f"{n:22}{r['CB513_Q3']:8.2f}{r['CB513_Q8']:8.2f}{r['CB513_SOV']:8.2f}" if r else f"{n:22}{'(run it)':>8}")
row('Full PSSP-ULTRA',full_ref); row('- ESM-2',noesm)
row('- CRF',res['noCRF']); row('- Transformer',res['noTransformer']); row('- TCN',res['noTCN'])
print('\nPaste these rows into the manuscript ablation table.')


=== ablation: noCRF ===
Downloading: "https://dl.fbaipublicfiles.com/fair-esm/models/esm2_t33_650M_UR50D.pt" to /root/.cache/torch/hub/checkpoints/esm2_t33_650M_UR50D.pt
Downloading: "https://dl.fbaipublicfiles.com/fair-esm/regression/esm2_t33_650M_UR50D-contact-regression.pt" to /root/.cache/torch/hub/checkpoints/esm2_t33_650M_UR50D-contact-regression.pt
[esm] using cache /content/drive/MyDrive/PSSP_ULTRA_DATASETS/esm_cache/esm_train_99ebc664.npy  (D=1280)
[esm] using cache /content/drive/MyDrive/PSSP_ULTRA_DATASETS/esm_cache/esm_cb513_4f9e0b20.npy  (D=1280)
[model] in_dim=1324  params=3.89M
[resume] no checkpoint -> training from scratch


/tmp/ipykernel_843/2477223729.py:164: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.norm_first was True
  self.tr = nn.TransformerEncoder(enc, n_tr)


ep001 loss   1.262  val Q3 82.84  Q8 68.58  SOV 80.81  (24s)
ep002 loss   0.796  val Q3 85.27  Q8 73.26  SOV 84.03  (24s)
ep003 loss   0.704  val Q3 86.21  Q8 75.32  SOV 84.28  (24s)
ep004 loss   0.662  val Q3 86.42  Q8 76.07  SOV 84.61  (24s)
ep005 loss   0.632  val Q3 86.13  Q8 76.48  SOV 84.06  (24s)
   [ckpt] saved through epoch 5 -> /content/drive/MyDrive/PSSP_ULTRA_DATASETS/runs/stable_v1_noCRF/ckpt_last.pt
ep006 loss   0.607  val Q3 86.83  Q8 77.17  SOV 84.76  (24s)
ep007 loss   0.582  val Q3 87.42  Q8 77.80  SOV 85.31  (24s)
ep008 loss   0.559  val Q3 87.76  Q8 78.02  SOV 85.91  (24s)
ep009 loss   0.538  val Q3 87.26  Q8 77.82  SOV 85.05  (24s)
ep010 loss   0.518  val Q3 87.47  Q8 77.91  SOV 86.11  (24s)
   [ckpt] saved through epoch 10 -> /content/drive/MyDrive/PSSP_ULTRA_DATASETS/runs/stable_v1_noCRF/ckpt_last.pt
ep011 loss   0.500  val Q3 87.51  Q8 77.92  SOV 85.91  (24s)
ep012 loss   0.482  val Q3 87.37  Q8 77.76  SOV 85.33  (25s)
ep013 loss   0.466  val Q3 87.53  Q8 77.83 

## 10. Confusion matrix (Fig. 4)
Full-model `best.pt` on CB513 -> `<outdir>/figures/fig4_confusion.pdf` + `per_class.json`.

In [ ]:
import torch, os, json, numpy as np
import matplotlib; matplotlib.use('Agg')
import matplotlib.pyplot as plt
from torch.utils.data import DataLoader
dev='cuda' if torch.cuda.is_available() else 'cpu'; OD=CONFIG['outdir']; FIGDIR=os.path.join(OD,'figures'); os.makedirs(FIGDIR,exist_ok=True)
te=load_npy_gz(CONFIG['test']); seq,valid=decode_sequences(te); y8,y3=q8_q3_labels(te,valid)
cache_dir=CONFIG['cache_dir'] or OD
te_esm,edim=(esm_cache(seq,'cb513',CONFIG['esm_model'],cache_dir,dev,CONFIG['esm_batch']) if CONFIG['use_esm'] else (None,0))
xl=DataLoader(PSSPSet(te,valid,y8,y3,te_esm,CONFIG['use_esm']),batch_size=CONFIG['batch'])
model=PSSPUltra(22+22+(edim if CONFIG['use_esm'] else 0),CONFIG['d_model'],CONFIG['n_tcn'],CONFIG['n_tr'],CONFIG['heads'],8,CONFIG['dropout']).to(dev)
model.load_state_dict(torch.load(os.path.join(OD,'best.pt'),map_location=dev,weights_only=False)); model.eval()
C=np.zeros((3,3),int)
with torch.no_grad():
    for x,v,yy8,_ in xl:
        for p,yt,m in zip(model.decode(x.to(dev),v.to(dev)),yy8,v):
            L=int(m.sum()); pr=Q8IDX_TO_Q3IDX[np.asarray(p[:L])]; gt=Q8IDX_TO_Q3IDX[yt[:L].numpy()]
            for a,b in zip(gt,pr): C[a,b]+=1
labels=['H','E','C']
pc={labels[c]:{'precision':round(100*C[c,c]/max(C[:,c].sum(),1),2),'recall':round(100*C[c,c]/max(C[c,:].sum(),1),2),'support':int(C[c,:].sum())} for c in range(3)}
for c in labels:
    a,b=pc[c]['precision'],pc[c]['recall']; pc[c]['f1']=round(2*a*b/max(a+b,1e-9),2)
json.dump(pc,open(os.path.join(FIGDIR,'per_class.json'),'w'),indent=2)
plt.rcParams.update({'font.size':9,'font.family':'sans-serif','savefig.dpi':300,'savefig.bbox':'tight'})
Cn=C/C.sum(1,keepdims=True); fig,ax=plt.subplots(figsize=(3.6,3.2)); im=ax.imshow(Cn,cmap='Blues',vmin=0,vmax=1)
ax.set_xticks(range(3)); ax.set_yticks(range(3)); ax.set_xticklabels(labels); ax.set_yticklabels(labels); ax.set_xlabel('Predicted'); ax.set_ylabel('True')
for i in range(3):
    for j in range(3): ax.text(j,i,f'{Cn[i,j]*100:.1f}',ha='center',va='center',color='white' if Cn[i,j]>0.5 else 'black',fontsize=8.5)
fig.colorbar(im,fraction=0.046,pad=0.04,label='row-normalised (%)')
for e in ('pdf','png'): fig.savefig(os.path.join(FIGDIR,f'fig4_confusion.{e}'))
plt.close(fig); print('saved', os.path.join(FIGDIR,'fig4_confusion.pdf'))

Downloading: "https://dl.fbaipublicfiles.com/fair-esm/models/esm2_t33_650M_UR50D.pt" to /root/.cache/torch/hub/checkpoints/esm2_t33_650M_UR50D.pt
Downloading: "https://dl.fbaipublicfiles.com/fair-esm/regression/esm2_t33_650M_UR50D-contact-regression.pt" to /root/.cache/torch/hub/checkpoints/esm2_t33_650M_UR50D-contact-regression.pt
[esm] using cache /content/drive/MyDrive/PSSP_ULTRA_DATASETS/esm_cache/esm_cb513_4f9e0b20.npy  (D=1280)


/tmp/ipykernel_788/2477223729.py:164: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.norm_first was True
  self.tr = nn.TransformerEncoder(enc, n_tr)


saved /content/drive/MyDrive/PSSP_ULTRA_DATASETS/runs/stable_v1/figures/fig4_confusion.pdf


## 10b. Table 4 — per-state precision / recall / F1 (paste-ready)
Prints Table 4 + ready-to-paste LaTeX rows (scikit-learn; manual fallback).

In [ ]:
import torch, os, json, numpy as np
from torch.utils.data import DataLoader
dev='cuda' if torch.cuda.is_available() else 'cpu'; OD=CONFIG['outdir']
te=load_npy_gz(CONFIG['test']); seq,valid=decode_sequences(te); y8,y3=q8_q3_labels(te,valid)
cache_dir=CONFIG['cache_dir'] or OD
te_esm,edim=(esm_cache(seq,'cb513',CONFIG['esm_model'],cache_dir,dev,CONFIG['esm_batch']) if CONFIG['use_esm'] else (None,0))
xl=DataLoader(PSSPSet(te,valid,y8,y3,te_esm,CONFIG['use_esm']),batch_size=CONFIG['batch'])
model=PSSPUltra(22+22+(edim if CONFIG['use_esm'] else 0),CONFIG['d_model'],CONFIG['n_tcn'],CONFIG['n_tr'],CONFIG['heads'],8,CONFIG['dropout']).to(dev)
model.load_state_dict(torch.load(os.path.join(OD,'best.pt'),map_location=dev,weights_only=False)); model.eval()
P,Tr=[],[]
with torch.no_grad():
    for x,v,yy8,_ in xl:
        for p,yt,m in zip(model.decode(x.to(dev),v.to(dev)),yy8,v):
            L=int(m.sum()); P.append(Q8IDX_TO_Q3IDX[np.asarray(p[:L])]); Tr.append(Q8IDX_TO_Q3IDX[yt[:L].numpy()])
P=np.concatenate(P); Tr=np.concatenate(Tr)
labels=['H','E','C']; names={'H':'Helix (H)','E':'Strand (E)','C':'Coil (C)'}
try:
    from sklearn.metrics import precision_recall_fscore_support
    pr,rc,f1,sup=precision_recall_fscore_support(Tr,P,labels=[0,1,2],zero_division=0)
except Exception:
    Cm=np.zeros((3,3),int)
    for a,b in zip(Tr,P): Cm[a,b]+=1
    pr=np.array([Cm[c,c]/max(Cm[:,c].sum(),1) for c in range(3)]); rc=np.array([Cm[c,c]/max(Cm[c,:].sum(),1) for c in range(3)]); f1=2*pr*rc/np.maximum(pr+rc,1e-9); sup=Cm.sum(1)
per={labels[c]:{'precision':round(100*pr[c],2),'recall':round(100*rc[c],2),'f1':round(100*f1[c],2),'support':int(sup[c])} for c in range(3)}
os.makedirs(os.path.join(OD,'figures'),exist_ok=True); json.dump(per,open(os.path.join(OD,'figures','per_class.json'),'w'),indent=2)
print('Table 4 -- per-state precision, recall, F1 (%) on CB513')
print(f"{'State':11}{'Precision':>10}{'Recall':>9}{'F1':>8}{'Support':>10}")
for c in labels:
    d=per[c]; print(f"{names[c]:11}{d['precision']:10.2f}{d['recall']:9.2f}{d['f1']:8.2f}{d['support']:10d}")
print(f"{'Macro avg':11}{100*pr.mean():10.2f}{100*rc.mean():9.2f}{100*f1.mean():8.2f}{int(sup.sum()):10d}")
print('\n--- paste-ready LaTeX rows ---')
for c in labels:
    d=per[c]; print(f"{names[c]:10} & {d['precision']:.2f} & {d['recall']:.2f} & {d['f1']:.2f} & {d['support']} \\\\")

[esm] using cache /content/drive/MyDrive/PSSP_ULTRA_DATASETS/esm_cache/esm_cb513_4f9e0b20.npy  (D=1280)


/tmp/ipykernel_788/2477223729.py:164: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.norm_first was True
  self.tr = nn.TransformerEncoder(enc, n_tr)


Table 4 -- per-state precision, recall, F1 (%) on CB513
State       Precision   Recall      F1   Support
Helix (H)       89.33    91.45   90.38     29648
Strand (E)      84.12    83.62   83.87     19089
Coil (C)        85.44    84.05   84.74     36028
Macro avg       86.30    86.37   86.33     84765

--- paste-ready LaTeX rows ---
Helix (H)  & 89.33 & 91.45 & 90.38 & 29648 \\
Strand (E) & 84.12 & 83.62 & 83.87 & 19089 \\
Coil (C)   & 85.44 & 84.05 & 84.74 & 36028 \\
